## Legal — Logs (Warehouse)
Audit logs → `WAREHOUSE`  |  Metadata logs → Fabric Plan DB

### DDL Reference (run once in Warehouse/Plan DB)

In [ ]:
# ── DDL reference (run once in Warehouse / Plan DB — do NOT execute in notebook) ──

# Warehouse  →  WAREHOUSE.Logs.legal_audit_logs
# CREATE TABLE Logs.legal_audit_logs (
#     audit_id           VARCHAR(36),
#     source_type        VARCHAR(200),
#     destination        VARCHAR(500),
#     notebook_name      VARCHAR(200),
#     layer_name         VARCHAR(100),
#     table_name         VARCHAR(200),
#     records_processed  BIGINT,
#     status             VARCHAR(50),
#     execution_time     DATETIME2(6),
#     error_message      VARCHAR(8000)
# );

# Warehouse  →  WAREHOUSE.Logs.legal_central_error_logs
# CREATE TABLE Logs.legal_central_error_logs (
#     Error_ID           VARCHAR(36),
#     Source_Table       VARCHAR(100),
#     Pipeline_Layer     VARCHAR(100),
#     Error_Message      VARCHAR(MAX),
#     Error_Record_JSON  VARCHAR(MAX),
#     Error_Logged_Time  DATETIME2(6)
# );

# Plan DB  →  __fabric_plan_sys-....Logs.metadata_logs
# CREATE TABLE Logs.metadata_logs (
#     metadata_id                    VARCHAR(36),
#     industry_vertical              VARCHAR(50),
#     source_type                    VARCHAR(50),
#     source_path                    VARCHAR(1000),
#     source_file_type               VARCHAR(50),
#     source_sheet_name              VARCHAR(255),
#     source_sheet_index             INT,
#     source_table_schema            VARCHAR(255),
#     source_table_name              VARCHAR(255),
#     file_name                      VARCHAR(255),
#     bronze_schema_name             VARCHAR(255),
#     bronze_table_name              VARCHAR(255),
#     bronze_created_at              DATETIME2(6),
#     silver_schema_name             VARCHAR(255),
#     silver_table_name              VARCHAR(255),
#     silver_created_at              DATETIME2(6),
#     gold_schema_name               VARCHAR(255),
#     gold_table_name                VARCHAR(255),
#     gold_created_at                DATETIME2(6),
#     table_type                     VARCHAR(50),
#     load_type                      VARCHAR(50),
#     incremental_key_column         VARCHAR(255),
#     last_updated_incremental_value VARCHAR(255),
#     frequency                      VARCHAR(100),
#     trigger_time                   VARCHAR(100),
#     pipeline_name                  VARCHAR(255),
#     parent_pipeline_name           VARCHAR(255),
#     child_pipeline_name            VARCHAR(255),
#     is_active                      VARCHAR(5),
#     last_updated_at                DATETIME2(6)
# );


### Connection Parameters

In [ ]:
import struct as _struct
import pyodbc

# ── Warehouse connection parameters ───────────────────────────────────────────
WAREHOUSE_SCHEMA       = "Logs"
WAREHOUSE_SERVER       = ("")
WAREHOUSE_DB           = ""
AUDIT_LOG_TABLE        = "legal_audit_logs"
CENTRAL_ERROR_LOG_WH   = "legal_central_error_logs"

# ── Plan DB connection parameters ────────────────────────────────────────────
PLAN_DB_SERVER         = ("")
PLAN_DB                = ""
PLAN_DB_SCHEMA         = "Logs"
METADATA_TABLE         = "metadata_logs"

print("Connection parameters set.")
print(f"  Warehouse : {WAREHOUSE_DB}  |  [{WAREHOUSE_SCHEMA}].[{AUDIT_LOG_TABLE}]")
print(f"  Plan DB   : {PLAN_DB}  |  [{PLAN_DB_SCHEMA}].[{METADATA_TABLE}]")


### Connection Helper Functions

In [ ]:
from datetime import datetime

def _get_token_struct():
    token        = notebookutils.credentials.getToken("https://database.windows.net/")
    token_bytes  = token.encode("utf-16-le")
    return _struct.pack(f'<I{len(token_bytes)}s', len(token_bytes), token_bytes)

SQL_COPT_SS_ACCESS_TOKEN = 1256

def get_warehouse_conn():
    """Returns an authenticated pyodbc connection to WAREHOUSE Warehouse."""
    conn_str = (
        "Driver={ODBC Driver 18 for SQL Server};"
        f"Server={WAREHOUSE_SERVER};"
        f"Database={WAREHOUSE_DB};"
        "Encrypt=yes;TrustServerCertificate=no;"
    )
    return pyodbc.connect(conn_str, attrs_before={SQL_COPT_SS_ACCESS_TOKEN: _get_token_struct()})


def get_plan_db_conn():
    """Returns an authenticated pyodbc connection to the Fabric Plan DB."""
    conn_str = (
        "Driver={ODBC Driver 18 for SQL Server};"
        f"Server={PLAN_DB_SERVER};"
        f"Database={PLAN_DB};"
        "Encrypt=yes;TrustServerCertificate=no;"
    )
    return pyodbc.connect(conn_str, attrs_before={SQL_COPT_SS_ACCESS_TOKEN: _get_token_struct()})


print("Connection helpers registered: get_warehouse_conn | get_plan_db_conn")


### `log_audit()` — Write to Warehouse Audit Log

In [ ]:
def log_audit(
    audit_id,
    source_type,
    destination,
    notebook_name,
    layer_name,
    table_name,
    records_processed,
    status,
    error_message=""
):
    """
    Appends one audit record to [{WAREHOUSE_SCHEMA}].[{AUDIT_LOG_TABLE}]
    in the Fabric Warehouse.
    """
    conn   = get_warehouse_conn()
    cursor = conn.cursor()
    cursor.execute(
        f"""
        INSERT INTO [{WAREHOUSE_SCHEMA}].[{AUDIT_LOG_TABLE}]
            (audit_id, source_type, destination, notebook_name, layer_name,
             table_name, records_processed, status, execution_time, error_message)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """,
        (
            audit_id,
            source_type,
            destination,
            notebook_name,
            layer_name,
            table_name,
            int(records_processed),
            status,
            datetime.now(),
            error_message,
        ),
    )
    conn.commit()
    conn.close()
    print(
        f"[AUDIT] {status} | notebook={notebook_name} | layer={layer_name} "
        f"| table={table_name} | records={records_processed}"
    )


print("log_audit() registered.")


### `upsert_bronze_metadata()` — Raw → Bronze metadata

In [ ]:
# ── Pipeline-level config (used by both metadata functions) ───────────────────
INDUSTRY_VERTICAL    = "Legal" # Example of Industry Vertical
SOURCE_TYPE          = "FILE"
FREQUENCY            = "Adhoc"
LOAD_TYPE            = "Full Load"
PIPELINE_NAME        = "PL_Legal_ETL"
PARENT_PIPELINE_NAME = "PL_Legal_ETL"


def upsert_bronze_metadata(file_name, table_name, file_path, extension, schema, metadata_id):
    """
    UPSERT into Plan DB metadata_logs for the Raw → Bronze layer.
    If file_name exists  → UPDATE bronze columns.
    If file_name absent  → INSERT full row.

    Parameters
    ----------
    file_name   : e.g. "raw_lawyer_profile.csv"
    table_name  : e.g. "raw_lawyer_profile"
    file_path   : full abfss:// source path
    extension   : e.g. "csv"  (uppercased internally)
    schema      : Bronze_Schema  →  "bronze_legal"
    metadata_id : str(uuid.uuid4())
    """
    try:
        conn   = get_plan_db_conn()
        cursor = conn.cursor()
        cursor.execute(
            f"""
            IF EXISTS (
                SELECT 1 FROM [{PLAN_DB_SCHEMA}].[{METADATA_TABLE}] WHERE file_name = ?
            )
                UPDATE [{PLAN_DB_SCHEMA}].[{METADATA_TABLE}]
                SET bronze_table_name    = ?,
                    bronze_schema_name   = ?,
                    source_path          = ?,
                    source_file_type     = ?,
                    frequency            = ?,
                    load_type            = ?,
                    pipeline_name        = ?,
                    parent_pipeline_name = ?,
                    bronze_created_at    = GETUTCDATE(),
                    last_updated_at      = GETUTCDATE(),
                    is_active            = 'Yes'
                WHERE file_name = ?
            ELSE
                INSERT INTO [{PLAN_DB_SCHEMA}].[{METADATA_TABLE}]
                (metadata_id, industry_vertical, source_type, source_path,
                 source_file_type, file_name, bronze_schema_name, bronze_table_name,
                 frequency, load_type, pipeline_name, parent_pipeline_name,
                 bronze_created_at, is_active, last_updated_at)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, GETUTCDATE(), 'Yes', GETUTCDATE())
            """,
            (
                # IF EXISTS check
                file_name,
                # UPDATE values
                table_name, schema, file_path, extension.upper(),
                FREQUENCY, LOAD_TYPE, PIPELINE_NAME, PARENT_PIPELINE_NAME,
                file_name,
                # INSERT values
                metadata_id, INDUSTRY_VERTICAL, SOURCE_TYPE, file_path,
                extension.upper(), file_name, schema, table_name,
                FREQUENCY, LOAD_TYPE, PIPELINE_NAME, PARENT_PIPELINE_NAME,
            ),
        )
        conn.commit()
        conn.close()
        print(f"[META] Bronze upserted | file={file_name} | table={schema}.{table_name}")
    except Exception as _me:
        print(f"[META] Bronze upsert FAILED | file={file_name} | error={_me}")
        raise


print("upsert_bronze_metadata() registered.")


### `update_layer_metadata()` — Bronze→Silver / Silver→Gold metadata

In [ ]:
def update_layer_metadata(layer, file_name, table_name, schema, table_type=None):
    """
    UPDATE Plan DB metadata_logs for Bronze→Silver or Silver→Gold.

    Parameters
    ----------
    layer      : "SILVER" or "GOLD"
    file_name  : source CSV file name  e.g. "raw_lawyer_profile.csv"
    table_name : target table name     e.g. "raw_lawyer_profile" / "Lawyer_Master_Data"
    schema     : Silver_Schema or Gold_Schema
    table_type : required for GOLD — "FACT" or "DIM"
    """
    try:
        conn   = get_plan_db_conn()
        cursor = conn.cursor()

        if layer == "SILVER":
            cursor.execute(
                f"""
                UPDATE [{PLAN_DB_SCHEMA}].[{METADATA_TABLE}]
                SET silver_table_name = ?,
                    silver_schema_name = ?,
                    silver_created_at  = GETUTCDATE(),
                    last_updated_at    = GETUTCDATE(),
                    is_active          = 'Yes'
                WHERE file_name = ?
                """,
                (table_name, schema, file_name),
            )

        elif layer == "GOLD":
            if not table_type:
                raise ValueError("table_type required for GOLD — pass 'FACT' or 'DIM'")
            cursor.execute(
                f"""
                UPDATE [{PLAN_DB_SCHEMA}].[{METADATA_TABLE}]
                SET gold_table_name  = ?,
                    gold_schema_name = ?,
                    table_type       = ?,
                    gold_created_at  = GETUTCDATE(),
                    last_updated_at  = GETUTCDATE(),
                    is_active        = 'Yes'
                WHERE file_name = ?
                """,
                (table_name, schema, table_type, file_name),
            )
        else:
            raise ValueError(f"Invalid layer '{layer}' — must be 'SILVER' or 'GOLD'")

        rows = cursor.rowcount
        conn.commit()
        conn.close()
        print(f"[META] {layer} updated | file={file_name} | table={schema}.{table_name} | rows={rows}")
        if rows == 0:
            print(f"[META WARNING] No rows matched file_name={file_name}")

    except Exception as _me:
        print(f"[META] {layer} update FAILED | file={file_name} | error={_me}")
        raise


print("update_layer_metadata() registered.")
